# Governance Orphans — EGA DAC continuity audit

Runs the full pipeline: census (`governance_decay_scanner.py`) then analysis (`study_analysis.py`).
Put both scripts, `consortium_roster.csv`, `validation_sample_labels.csv` and `requirements.txt` in the working directory.
See `README.md` for what each output means.

In [ ]:
!pip install -q -r requirements.txt

## 1. Census
Set `LIVE = True` to fetch the whole archive from EGA (~45 min, prints progress continuously).
With `LIVE = False` it re-analyses an existing `raw_ega_snapshot/` (must be present locally).

In [ ]:
LIVE = False
import subprocess, sys
cmd = [sys.executable, '-u', 'governance_decay_scanner.py'] + ([] if LIVE else ['--cached'])
print(subprocess.run(cmd, capture_output=True, text=True).stdout[-6000:])

## 2. Validation, statistics, sensitivity, continuity checks

In [ ]:
import json
r = subprocess.run([sys.executable, 'study_analysis.py'], capture_output=True, text=True)
print(r.stderr[-2000:])
res = json.load(open('results/study_results.json'))
print(json.dumps(res['headline'], indent=2))

In [ ]:
import pandas as pd
display(pd.read_csv('results/defects_by_cohort.csv'))
display(pd.read_csv('results/sensitivity.csv', index_col=0))

## 3. Figures

In [ ]:
from IPython.display import Image, display
for f in ['fig_orphan_impact.png', 'fig_defects_by_cohort.png', 'fig_continuity_checklist.png',
          'chart_reachability.png', 'chart_top15_by_impact.png']:
    display(Image(f))

## 4. Tests and the continuity checker

In [ ]:
r = subprocess.run([sys.executable, '-m', 'pytest', '-q'], capture_output=True, text=True)
print(r.stdout[-1500:])

In [ ]:
print(subprocess.run([sys.executable, 'check_dac.py', 'EGAC00001000010', 'EGAC00000000008'], capture_output=True, text=True).stdout)